# p01 · The jump-off: an invertible map between locations and pages
Basile's `libraryofbabel.info`: a location is a number, a page is a base-29 numeral, and the two are joined by an **invertible pseudo-random number generator** ("a pseudo-random number generating algorithm … without needing to store anything on disk … for any block of text, the program can work backwards to calculate its location", theory page 4). The published code (`librarianofbabel/libraryofbabel.info-algo`, CC BY-SA-NC, uploaded 2019-01-02) is the image library's; its README gives the recipe: an LCG `pointer = (a·pointer + c) mod m`, then shift/XOR steps, and the inverse by `a⁻¹` and the shifts undone. This notebook realises that **recipe** independently at a size that can be checked exhaustively.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
# Listing 1 - the ordered address (bijective base-N): text <-> non-negative integer
def address(text, alphabet):
    """Rank of `text` among all strings over `alphabet`, shortest first, then alphabetical."""
    N, a = len(alphabet), 0
    for ch in text:
        a = a * N + alphabet.index(ch) + 1
    return a - 1

def text_at(a, alphabet):
    """Inverse of address(): the string whose rank is `a`."""
    N, out = len(alphabet), []
    a += 1
    while a > 0:
        a, r = divmod(a - 1, N)
        out.append(alphabet[r])
    return "".join(reversed(out))

def page_at(loc, alphabet, k):
    """Fixed-length page: the k-digit base-N numeral of `loc` (digit 0 = first symbol)."""
    N, out = len(alphabet), []
    for _ in range(k):
        loc, r = divmod(loc, N)
        out.append(alphabet[r])
    return "".join(reversed(out))

def loc_of(page, alphabet):
    """Inverse of page_at()."""
    N, loc = len(alphabet), 0
    for ch in page:
        loc = loc * N + alphabet.index(ch)
    return loc

# Listing 2 - an invertible scrambler of [0, N): LCG + xorshift, cycle-walked into range
def make_scrambler(N, a=1664525, c=1013904223, s1=7, s2=5):
    e = max(1, (N - 1).bit_length()); M = 1 << e; mask = M - 1
    a_inv = pow(a, -1, M)                       # a odd -> invertible mod 2^e
    def f(x):                                   # bijection on [0, M)
        x = (a * x + c) & mask                  # Hull-Dobell: a = 1 mod 4, c odd
        x ^= x >> s1
        x ^= (x << s2) & mask
        return x
    def finv(y):
        z = y
        for _ in range(e // s2 + 1): z = y ^ ((z << s2) & mask)
        y = z
        for _ in range(e // s1 + 1): z = y ^ (z >> s1)
        return (a_inv * (z - c)) & mask
    def scramble(x):                            # bijection on [0, N)
        x = f(x)
        while x >= N: x = f(x)
        return x
    def unscramble(y):
        y = finv(y)
        while y >= N: y = finv(y)
        return y
    return scramble, unscramble

print('listings defined')

listings defined


## The toy library: 29 symbols, pages of 3 characters (N = 29³ = 24,389 pages)
The scrambler is checked on **every** location, not sampled: it is a bijection on [0, N) and the inverse is exact.

In [3]:
ALPHA = "abcdefghijklmnopqrstuvwxyz ,."; K = 3; N = len(ALPHA) ** K
scramble, unscramble = make_scrambler(N)
seen = set()
for loc in range(N):
    y = scramble(loc); assert 0 <= y < N and unscramble(y) == loc; seen.add(y)
print("N =", N, "| distinct scrambled values:", len(seen), "| every location round-trips: True")
assert len(seen) == N

N = 24389 | distinct scrambled values: 24389 | every location round-trips: True


## A page and its location, both directions
`page` = the page whose location is `loc` in the *scrambled* library; `ordered` = the page at the same location in the ordered library (Listing 1's numeral).

In [4]:
def babel_page(loc):  return page_at(scramble(loc), ALPHA, K)
def babel_loc(page):  return unscramble(loc_of(page, ALPHA))
print("loc    scrambled page   ordered page")
for loc in range(12): print("%-6d %-16r %r" % (loc, babel_page(loc), page_at(loc, ALPHA, K)))
for loc in (0, 1, 777, N - 1): assert babel_loc(babel_page(loc)) == loc
print("babel_loc(babel_page(loc)) == loc  for sampled locations; exhaustive check above")

loc    scrambled page   ordered page
0      'blr'            'aaa'
1      'lrq'            'aab'
2      'wfl'            'aac'
3      'is,'            'aad'
4      'did'            'aae'
5      ',dz'            'aaf'
6      'qxx'            'aag'
7      'bw '            'aah'
8      'zmd'            'aai'
9      'o, '            'aaj'
10     'cvl'            'aak'
11     'zns'            'aal'
babel_loc(babel_page(loc)) == loc  for sampled locations; exhaustive check above


## Neighbouring locations: what each numbering does with them
Mean number of the 3 characters that differ between the pages at locations L and L + 1, over all 24,388 neighbouring pairs.

In [5]:
def diff(a, b): return sum(x != y for x, y in zip(a, b))
ordered = sum(diff(page_at(L, ALPHA, K), page_at(L + 1, ALPHA, K)) for L in range(N - 1)) / (N - 1)
scrambled = sum(diff(babel_page(L), babel_page(L + 1)) for L in range(N - 1)) / (N - 1)
print("ordered   : %.3f characters differ between neighbours" % ordered)
print("scrambled : %.3f characters differ between neighbours  (independent pages would give %.3f)" % (scrambled, 3 * 28 / 29))
assert ordered < 1.1 and scrambled > 2.5

ordered   : 1.036 characters differ between neighbours
scrambled : 2.910 characters differ between neighbours  (independent pages would give 2.897)


## Same set, same information
The scrambler is a permutation of [0, N). Both libraries hold the same N pages and a location costs the same log₂N bits in each; only the *order* differs. Which page sits where is what the scrambler decides; **the address of a page is computed from the page in both, and nothing is stored.**

In [6]:
print("log2(N) = %.2f bits per location in both libraries" % math.log2(N))
print("table entries stored by either library: 0 (scramble/unscramble are O(log N) arithmetic)")

log2(N) = 14.57 bits per location in both libraries
table entries stored by either library: 0 (scramble/unscramble are O(log N) arithmetic)
